# <b> Evaluatie transcriptie en sprekerdiarisatie van het Medium Whisper model </b>

Geteste modellen:
- medium
- large-v3 alleen bij twijfel / als referentie

Per fragment vergelijk ik:
1. handmatige referentietranscriptie
2. output van Whisper
3. opvallende fouten
4. mogelijke impact op didactische analyse

## 1. Functies

#### 1.1. Bestanden laden

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML
import html


def load_diarized_json(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data["segments"]), data


def load_manual_transcript(path):
    path = Path(path)

    lines = [
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    data = []

    for line in lines:
        if ":" not in line:
            continue

        speaker, text = line.split(":", 1)

        data.append({
            "speaker": speaker.strip(),
            "text": text.strip().strip('"')
        })

    return pd.DataFrame(data)

#### 1.2. Vergelijken van transcripties

In [2]:
def show_transcription_comparison(
    df,
    manual_df,
    start_time,
    end_time,
    title="Handmatige transcriptie vs. Whisper"
):
    fragment = df[
        (df["end"] >= start_time) &
        (df["start"] <= end_time)
    ].copy()

    manual_text = "\n\n".join(manual_df["text"].tolist())
    whisper_text = "\n\n".join(fragment["text"].tolist())

    display(HTML(f"""
    <h3>{title}</h3>

    <table style="width:100%; table-layout:fixed; border-collapse:collapse;">
        <tr>
            <th style="width:50%; text-align:left; padding:10px; border:1px solid #ccc;">
                Handmatig
            </th>
            <th style="width:50%; text-align:left; padding:10px; border:1px solid #ccc;">
                Whisper
            </th>
        </tr>
        <tr>
            <td style="vertical-align:top; padding:12px; border:1px solid #ccc; white-space:pre-wrap;">
                {html.escape(manual_text)}
            </td>
            <td style="vertical-align:top; padding:12px; border:1px solid #ccc; white-space:pre-wrap;">
                {html.escape(whisper_text)}
            </td>
        </tr>
    </table>
    """))

    return fragment

#### 1.3. Vergelijken van sprekerdiarisatie 

In [3]:
def show_diarization_comparison(
    df,
    manual_df,
    start_time,
    end_time
):
    fragment = df[
        (df["end"] >= start_time) &
        (df["start"] <= end_time)
    ].copy()

    manual_text = "\n\n".join(
        f"{row['speaker']}: {row['text']}"
        for _, row in manual_df.iterrows()
    )

    diarization_text = "\n\n".join(
        f"[{row['start']:.1f}-{row['end']:.1f}] "
        f"{row['speaker']}: {row['text']}"
        for _, row in fragment.iterrows()
    )

    display(HTML(f"""
    <h3>Handmatige sprekerverdeling vs. diarization</h3>

    <table style="width:100%; table-layout:fixed; border-collapse:collapse;">
        <tr>
            <th style="width:50%; text-align:left; padding:10px; border:1px solid #ccc;">
                Handmatig
            </th>
            <th style="width:50%; text-align:left; padding:10px; border:1px solid #ccc;">
                Diarization
            </th>
        </tr>
        <tr>
            <td style="vertical-align:top; padding:12px; border:1px solid #ccc; white-space:pre-wrap;">
                {html.escape(manual_text)}
            </td>
            <td style="vertical-align:top; padding:12px; border:1px solid #ccc; white-space:pre-wrap;">
                {html.escape(diarization_text)}
            </td>
        </tr>
    </table>
    """))

    details = fragment[
        [
            "start",
            "end",
            "speaker",
            "text",
            "speaker_confidence",
            "overlap",
            "uncertain_assignment"
        ]
    ].copy()

    details["start"] = details["start"].round(1)
    details["end"] = details["end"].round(1)
    details["speaker_confidence"] = details["speaker_confidence"].round(2)

    display(details)

    return fragment

## <b> 2. Fragment 1 — rumoer + overlap </b>

Duur: 45 sec  
Model: medium  
Bestand: `testaudio1_fragment.mp3`



## 2.1. Handmatige transcriptie versus Whisper transcriptie

In [4]:
df1, data1 = load_diarized_json(
    "../../Data-local/processed/diarization/testaudio1_fragment_diarized.json"
)

manual1 = load_manual_transcript(
    "../../Data-local/raw/testaudio1_manual.txt"
)

show_transcription_comparison(
    df1,
    manual1,
    start_time=5,
    end_time=21
)

Handmatig,Whisper
"Okee, heel mooi. Dus dit ben jij? Dit ben ik. Zit er ook nog in je karaktereigenschap met opruimen iets? Nee. Nee. Dit is wel een hele originele manier om te laten zien wie je bent Ja Zitten alle ingredienten in. Wie wil hem nog opeten?","Oké, heel mooi. Dit ben jij. Dit ben ik. Zit er ook nog in je karaktereigenschap iets met opruimen? Nee. Dit is een heel originele manier om te laten zien wie je bent. Er zitten alle ingrediënten in. Wie wil er nog opeten?"


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
2,4.12,9.36,MAIN_SPEAKER,"Oké, heel mooi.",-0.42402,0.067625,1.470588,[],SPEAKER_01,0.5615,True,True
3,9.52,11.64,MAIN_SPEAKER,Dit ben jij. Dit ben ik.,-0.42402,0.067625,1.470588,[],SPEAKER_01,0.7377,True,False
4,11.80,15.64,MAIN_SPEAKER,Zit er ook nog in je karaktereigenschap iets m...,-0.42402,0.067625,1.470588,[],SPEAKER_01,0.9010,False,False
5,15.80,20.28,MAIN_SPEAKER,Nee. Dit is een heel originele manier om te la...,-0.42402,0.067625,1.470588,[],SPEAKER_01,1.0000,False,False
6,20.44,24.48,OTHER_SPEAKER_1,Er zitten alle ingrediënten in. Wie wil er nog...,-0.42402,0.067625,1.470588,[],SPEAKER_00,0.5329,True,True


### 2.1. Conclusie transcriptie

De transcriptie met Whisper Medium behoudt de inhoud van het gesprek grotendeels correct. 
Enkele korte leerlingreacties worden samengevoegd met docenttekst of niet afzonderlijk weergegeven. 
Ook wijkt de formulering op sommige plekken licht af van de handmatige transcriptie, zonder dat de hoofdboodschap verloren gaat.

Voor analyse van docentgedrag lijkt deze transcriptkwaliteit voorlopig bruikbaar. 
Voor indicatoren die afhankelijk zijn van korte leerlingreacties of exacte interactievolgorde blijft extra voorzichtigheid nodig.

## 2.2. Sprekerdiarisatie-evaluatie

In [5]:
show_diarization_comparison(
    df1,
    manual1,
    start_time=5,
    end_time=21
)

Handmatig,Diarization
"Docent: Okee, heel mooi. Dus dit ben jij? Leerling: Dit ben ik. Docent: Zit er ook nog in je karaktereigenschap met opruimen iets? Leerling: Nee. Nee. Docent: Dit is wel een hele originele manier om te laten zien wie je bent Leerling: Ja Docent: Zitten alle ingredienten in. Leerling: Wie wil hem nog opeten?","[4.1-9.4] MAIN_SPEAKER: Oké, heel mooi. [9.5-11.6] MAIN_SPEAKER: Dit ben jij. Dit ben ik. [11.8-15.6] MAIN_SPEAKER: Zit er ook nog in je karaktereigenschap iets met opruimen? [15.8-20.3] MAIN_SPEAKER: Nee. Dit is een heel originele manier om te laten zien wie je bent. [20.4-24.5] OTHER_SPEAKER_1: Er zitten alle ingrediënten in. Wie wil er nog opeten?"


,start,end,speaker,text,speaker_confidence,overlap,uncertain_assignment
2,4.1,9.4,MAIN_SPEAKER,"Oké, heel mooi.",0.56,True,True
3,9.5,11.6,MAIN_SPEAKER,Dit ben jij. Dit ben ik.,0.74,True,False
4,11.8,15.6,MAIN_SPEAKER,Zit er ook nog in je karaktereigenschap iets m...,0.90,False,False
5,15.8,20.3,MAIN_SPEAKER,Nee. Dit is een heel originele manier om te la...,1.00,False,False
6,20.4,24.5,OTHER_SPEAKER_1,Er zitten alle ingrediënten in. Wie wil er nog...,0.53,True,True


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
2,4.12,9.36,MAIN_SPEAKER,"Oké, heel mooi.",-0.42402,0.067625,1.470588,[],SPEAKER_01,0.5615,True,True
3,9.52,11.64,MAIN_SPEAKER,Dit ben jij. Dit ben ik.,-0.42402,0.067625,1.470588,[],SPEAKER_01,0.7377,True,False
4,11.80,15.64,MAIN_SPEAKER,Zit er ook nog in je karaktereigenschap iets m...,-0.42402,0.067625,1.470588,[],SPEAKER_01,0.9010,False,False
5,15.80,20.28,MAIN_SPEAKER,Nee. Dit is een heel originele manier om te la...,-0.42402,0.067625,1.470588,[],SPEAKER_01,1.0000,False,False
6,20.44,24.48,OTHER_SPEAKER_1,Er zitten alle ingrediënten in. Wie wil er nog...,-0.42402,0.067625,1.470588,[],SPEAKER_00,0.5329,True,True


### 2.2.1. Diarization conclusie

De diarization herkent de hoofdspreker in delen van het fragment goed, maar korte wisselingen tussen docent en leerling worden niet betrouwbaar gescheiden. Een belangrijke oorzaak is dat sommige Whisper-segmenten al tekst van meerdere sprekers bevatten. Omdat de huidige pipeline één sprekerlabel per Whisper-segment toekent, kan een korte leerlingreactie daardoor aan de docent worden gekoppeld of andersom.  
De diarization is daarom bruikbaar om langere spreekbeurten en de dominante spreker te onderscheiden, maar minder betrouwbaar voor zeer korte interacties en snelle sprekerwisselingen. Dit wordt als beperking meegenomen in de latere classificatie.

## <b> 3. Fragment 2 — rumoer + overlap </b>

Duur: 45 sec  
Model: medium  
Bestand: `testaudio2_fragment.mp3`

## 3.1. Handmatige transcriptie versus Whisper transcriptie

In [6]:
df2, data2 = load_diarized_json(
    "../../Data-local/processed/diarization/testaudio2_fragment_diarized.json"
)

manual2 = load_manual_transcript(
    "../../Data-local/raw/testaudio2_manual.txt"
)

show_transcription_comparison(
    df2,
    manual2,
    start_time=0,
    end_time=30
)

Handmatig,Whisper
"Zegt u het maar. Ja, kijk maar even in je schrift. Opdracht vier. Uh, Ruud heeft een brommer. De tekst gaat toch niet over brommer, of wel? Nee, maar het gaat ook niet om de brommer. Maar Ruud heeft een probleem. Dat is de vraag. Wat voor een soort probleem heeft die? Ja, uh. Met z'n vrouw. Nee, klopt niet. Ik bedoel met z'n hond Ja, je bent gewoon verzinnen, of niet? Nee Jawel. Dat staat hier. Echt waar. Nee, ja het staat in jouw schrift, maar dat- (moeilijk te volgen), ja misschien staat het niet eens in je schrift. Misschien zit het gewoon te verzinnen. Nee, dat kan niet. Ja, ik zie jouw ogen steeds heen en weer gaan, dus ik denk, uh- nou gaat je hoofd heen en weer.","Vier, Rick. Zegt u het maar. Ja, kijk maar even in je schrift op de vier. Ruud heeft een brommer. De tekst gaat toch niet over brommer, of wel? Nee, maar het gaat ook niet om de brommer. Maar Ruud heeft een probleem, dat is de vraag. Wat voor probleem heeft-ie? Met zijn vrouw. Nee, klopt niet. Ik bedoel, met zijn hond. Ja, je bent gewoon verzinnen, of niet? Nee. Jawel. Dan staat-ie echt waar. Ja, het staat in jouw schrift. Misschien staat het niet in zijn schrift, misschien is-ie het gewoon te verzinnen. Je ziet je ogen heen en weer gaan. Dus ik denk, nou gaat je hoofd heen en weer. Dus ja, ja."


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
0,0.0,2.0,MAIN_SPEAKER,"Vier, Rick. Zegt u het maar.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
1,2.0,4.0,MAIN_SPEAKER,"Ja, kijk maar even in je schrift op de vier.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.7338,True,False
2,4.0,5.0,MAIN_SPEAKER,Ruud heeft een brommer.,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
3,5.0,7.0,MAIN_SPEAKER,"De tekst gaat toch niet over brommer, of wel?",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
4,7.0,9.0,MAIN_SPEAKER,"Nee, maar het gaat ook niet om de brommer.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.7548,True,False
5,9.0,11.0,MAIN_SPEAKER,"Maar Ruud heeft een probleem, dat is de vraag.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.6511,True,False
6,11.0,13.0,MAIN_SPEAKER,Wat voor probleem heeft-ie?,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
7,13.0,14.0,MAIN_SPEAKER,Met zijn vrouw.,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
8,14.0,15.0,MAIN_SPEAKER,"Nee, klopt niet.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
9,15.0,16.0,MAIN_SPEAKER,"Ik bedoel, met zijn hond.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False


### 3.1.1. Conclusie transcriptie

Whisper Medium presteert in dit fragment goed. De inhoud en volgorde van het gesprek worden grotendeels correct weergegeven en zowel de vragen van de docent als de antwoorden van de leerling blijven inhoudelijk herkenbaar. Er zijn kleine verschillen in formulering, maar deze veranderen de betekenis nauwelijks.  
Voor didactische analyse is deze transcriptkwaliteit voldoende om bijvoorbeeld vraaggedrag, uitleg en inhoudelijke reacties van de docent te herkennen.

## 3.2. Diarization-evaluatie

In [7]:
show_diarization_comparison(
    df2,
    manual2,
    start_time=0,
    end_time=30
)

Handmatig,Diarization
"Docent: Zegt u het maar. Ja, kijk maar even in je schrift. Opdracht vier. Leerling: Uh, Ruud heeft een brommer. Docent: De tekst gaat toch niet over brommer, of wel? Leerling: Nee, maar het gaat ook niet om de brommer. Docent: Maar Ruud heeft een probleem. Dat is de vraag. Wat voor een soort probleem heeft die? Leerling: Ja, uh. Met z'n vrouw. Docent: Nee, klopt niet. Leerling: Ik bedoel met z'n hond Docent: Ja, je bent gewoon verzinnen, of niet? Leerling: Nee Docent: Jawel. Leerling: Dat staat hier. Echt waar. Docent: Nee, ja het staat in jouw schrift, maar dat- (moeilijk te volgen), ja misschien staat het niet eens in je schrift. Misschien zit het gewoon te verzinnen. Leerling: Nee, dat kan niet. Docent: Ja, ik zie jouw ogen steeds heen en weer gaan, dus ik denk, uh- nou gaat je hoofd heen en weer.","[0.0-2.0] MAIN_SPEAKER: Vier, Rick. Zegt u het maar. [2.0-4.0] MAIN_SPEAKER: Ja, kijk maar even in je schrift op de vier. [4.0-5.0] MAIN_SPEAKER: Ruud heeft een brommer. [5.0-7.0] MAIN_SPEAKER: De tekst gaat toch niet over brommer, of wel? [7.0-9.0] MAIN_SPEAKER: Nee, maar het gaat ook niet om de brommer. [9.0-11.0] MAIN_SPEAKER: Maar Ruud heeft een probleem, dat is de vraag. [11.0-13.0] MAIN_SPEAKER: Wat voor probleem heeft-ie? [13.0-14.0] MAIN_SPEAKER: Met zijn vrouw. [14.0-15.0] MAIN_SPEAKER: Nee, klopt niet. [15.0-16.0] MAIN_SPEAKER: Ik bedoel, met zijn hond. [16.0-17.0] MAIN_SPEAKER: Ja, je bent gewoon verzinnen, of niet? [17.0-18.0] MAIN_SPEAKER: Nee. [18.0-19.0] MAIN_SPEAKER: Jawel. [19.0-20.0] MAIN_SPEAKER: Dan staat-ie echt waar. [20.0-21.0] MAIN_SPEAKER: Ja, het staat in jouw schrift. [21.0-23.0] MAIN_SPEAKER: Misschien staat het niet in zijn schrift, misschien is-ie het gewoon te verzinnen. [23.0-25.0] MAIN_SPEAKER: Je ziet je ogen heen en weer gaan. [25.0-28.0] MAIN_SPEAKER: Dus ik denk, nou gaat je hoofd heen en weer. [28.0-30.0] MAIN_SPEAKER: Dus ja, ja."


,start,end,speaker,text,speaker_confidence,overlap,uncertain_assignment
0,0.0,2.0,MAIN_SPEAKER,"Vier, Rick. Zegt u het maar.",1.00,False,False
1,2.0,4.0,MAIN_SPEAKER,"Ja, kijk maar even in je schrift op de vier.",0.73,True,False
2,4.0,5.0,MAIN_SPEAKER,Ruud heeft een brommer.,1.00,False,False
3,5.0,7.0,MAIN_SPEAKER,"De tekst gaat toch niet over brommer, of wel?",1.00,False,False
4,7.0,9.0,MAIN_SPEAKER,"Nee, maar het gaat ook niet om de brommer.",0.75,True,False
5,9.0,11.0,MAIN_SPEAKER,"Maar Ruud heeft een probleem, dat is de vraag.",0.65,True,False
6,11.0,13.0,MAIN_SPEAKER,Wat voor probleem heeft-ie?,1.00,False,False
7,13.0,14.0,MAIN_SPEAKER,Met zijn vrouw.,1.00,False,False
8,14.0,15.0,MAIN_SPEAKER,"Nee, klopt niet.",1.00,False,False
9,15.0,16.0,MAIN_SPEAKER,"Ik bedoel, met zijn hond.",1.00,False,False


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
0,0.0,2.0,MAIN_SPEAKER,"Vier, Rick. Zegt u het maar.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
1,2.0,4.0,MAIN_SPEAKER,"Ja, kijk maar even in je schrift op de vier.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.7338,True,False
2,4.0,5.0,MAIN_SPEAKER,Ruud heeft een brommer.,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
3,5.0,7.0,MAIN_SPEAKER,"De tekst gaat toch niet over brommer, of wel?",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
4,7.0,9.0,MAIN_SPEAKER,"Nee, maar het gaat ook niet om de brommer.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.7548,True,False
5,9.0,11.0,MAIN_SPEAKER,"Maar Ruud heeft een probleem, dat is de vraag.",-0.230623,0.074439,1.852201,[],SPEAKER_00,0.6511,True,False
6,11.0,13.0,MAIN_SPEAKER,Wat voor probleem heeft-ie?,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
7,13.0,14.0,MAIN_SPEAKER,Met zijn vrouw.,-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
8,14.0,15.0,MAIN_SPEAKER,"Nee, klopt niet.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False
9,15.0,16.0,MAIN_SPEAKER,"Ik bedoel, met zijn hond.",-0.230623,0.074439,1.852201,[],SPEAKER_00,1.0000,False,False


### 3.2.1. Diarization conclusie

De diarization is in dit fragment onvoldoende voor betrouwbare sprekerverdeling. Hoewel de handmatige transcriptie duidelijke wisselingen tussen docent en leerling bevat, worden vrijwel alle transcriptsegmenten als MAIN_SPEAKER geclassificeerd.  
Dit betekent dat de tekstinhoud bruikbaar is, maar dat niet betrouwbaar kan worden aangenomen dat iedere als MAIN_SPEAKER gelabelde uitspraak daadwerkelijk van de docent afkomstig is. Voor analyses die alleen naar de inhoud van langere docentpassages kijken kan het fragment nog bruikbaar zijn, maar tellingen van docent- versus leerlinginteractie zouden hierdoor vertekend kunnen worden.

## <b> 4. Fragment 3 — meerdere sprekers + rumoer </b>

Duur: 46 sec  
Model: medium  
Bestand: `testaudio5_fragment.mp3`

## 4.1. Handmatige transcriptie versus Whisper transcriptie

In [8]:
df3, data3 = load_diarized_json(
    "../../Data-local/processed/diarization/testaudio5_fragment_diarized.json"
)

manual3 = load_manual_transcript(
    "../../Data-local/raw/testaudio5_manual.txt"
)

show_transcription_comparison(
    df3,
    manual3,
    start_time=0,
    end_time=46
)

Handmatig,Whisper
"Oswaldo is je beste vriend, he? Of niet? Hier in de klas, ja. Stel je nou voor dat Oswaldo, uh, dat ie homo is. Zou dat invloed hebben op je vriendschap? Jazeker. Dat zou wel invloed hebben, ja Wat wil je? Ja, gewoon, toch? Het heeft invloed op onze vriendschap. Als ik zeg dat ik homo ben, dan ga je niet meer met me om? Ja, daar komt het wel op neer, ja. What the fuck? Je bent naar m'n huis geweest en alles en al. Dan ga je nu zeggen dat je homo bent, dan gaan mensen denken dat ik ook- omdat ik met jou nu- (slecht te verstaan) je zegt nu 'ik ben met jou naar huis geweest. Maar dat ligt gewoon aan jou, niet aan die mensen. Dan ligt het- Op dat moment ligt het aan jou, want jij bent dan homo. Wow, wacht even. Wat is er aan de hand?","Oswaldo is jouw beste vriend, hè? Of niet? Hier in de klas, ja. Stel je nou voor dat Oswaldo homo is. Zou dat invloed hebben op je vriendschap? Jazeker. Dat zou wel invloed hebben. Wat bedoel je? Gewoon, toch? Dat heeft invloed op onze vriendschap. Als ik zeg dat ik homo ben, dan ga je niet meer met me om. Dan komt het wel op neer, ja. Wat is er met me huis geweest? Als jij nu zegt dat je homo bent, dan gaan mensen denken dat ik ook... Omdat ik met jou nu, Safi, zeg nu, ik ben met jou naar huis geweest. Kere! Kere! Dat krijgen we dan nooit doen! Maar dat ligt gewoon aan jou, niet aan die mensen. Nee, dan ligt het op dat moment aan jou, want jij bent dan homo. Dat ligt gewoon aan jou, want jij bent dan homo. Wacht even, wat is er aan de hand? Hoe is het een facky..."


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
0,0.82,3.26,MAIN_SPEAKER,"Oswaldo is jouw beste vriend, hè? Of niet?",-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
1,3.38,4.38,MAIN_SPEAKER,"Hier in de klas, ja.",-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
2,4.50,8.98,MAIN_SPEAKER,Stel je nou voor dat Oswaldo homo is.,-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
3,9.10,10.82,MAIN_SPEAKER,Zou dat invloed hebben op je vriendschap?,-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
4,10.94,12.54,MAIN_SPEAKER,Jazeker. Dat zou wel invloed hebben.,-0.33186,0.043011,1.698842,[],SPEAKER_02,0.9385,False,False
5,12.66,13.66,OTHER_SPEAKER_4,Wat bedoel je?,-0.33186,0.043011,1.698842,[],SPEAKER_04,0.9859,False,False
6,13.78,14.74,OTHER_SPEAKER_3,"Gewoon, toch?",-0.33186,0.043011,1.698842,[],SPEAKER_01,1.0000,False,False
7,14.86,17.22,OTHER_SPEAKER_4,Dat heeft invloed op onze vriendschap.,-0.33186,0.043011,1.698842,[],SPEAKER_04,1.0000,False,False
8,17.34,20.54,OTHER_SPEAKER_3,"Als ik zeg dat ik homo ben, dan ga je niet mee...",-0.33186,0.043011,1.698842,[],SPEAKER_01,1.0000,False,False
9,20.66,22.82,OTHER_SPEAKER_3,"Dan komt het wel op neer, ja.",-0.33186,0.043011,1.698842,[],SPEAKER_01,0.9516,False,True


### 4.1.1. Conclusie transcriptie

Ook in dit complexere fragment blijft de hoofdinhoud van het gesprek grotendeels herkenbaar. Whisper registreert de belangrijkste uitspraken en de globale ontwikkeling van het gesprek correct. Bij rumoer, overlap en snel wisselende sprekers ontstaan echter duidelijk meer fouten en moeilijk interpreteerbare tekst.  
Dit fragment laat zien dat Whisper Medium voldoende context kan behouden voor globale analyse, maar dat afzonderlijke uitspraken in chaotische passages niet zonder meer als betrouwbaar mogen worden beschouwd. Voor didactische analyse kunnen duidelijke docentinterventies wel worden gebruikt, terwijl details uit overlappende leerlinggesprekken voorzichtiger geïnterpreteerd moeten worden.

## 4.2. Sprekerdiarisatie-evaluatie

In [9]:
show_diarization_comparison(
    df3,
    manual3,
    start_time=0,
    end_time=45
)

Handmatig,Diarization
"Docent: Oswaldo is je beste vriend, he? Of niet? Hier in de klas, ja. Stel je nou voor dat Oswaldo, uh, dat ie homo is. Zou dat invloed hebben op je vriendschap? Leerling 1: Jazeker. Dat zou wel invloed hebben, ja Leerling 2: Wat wil je? Leerling 1: Ja, gewoon, toch? Het heeft invloed op onze vriendschap. Leerling 2: Als ik zeg dat ik homo ben, dan ga je niet meer met me om? Leerling 1: Ja, daar komt het wel op neer, ja. Leerling 2: What the fuck? Je bent naar m'n huis geweest en alles en al. Leerling 1: Dan ga je nu zeggen dat je homo bent, dan gaan mensen denken dat ik ook- omdat ik met jou nu- (slecht te verstaan) je zegt nu 'ik ben met jou naar huis geweest. Leerling 2: Maar dat ligt gewoon aan jou, niet aan die mensen. Leerling 1: Dan ligt het- Op dat moment ligt het aan jou, want jij bent dan homo. Docent: Wow, wacht even. Wat is er aan de hand?","[0.8-3.3] MAIN_SPEAKER: Oswaldo is jouw beste vriend, hè? Of niet? [3.4-4.4] MAIN_SPEAKER: Hier in de klas, ja. [4.5-9.0] MAIN_SPEAKER: Stel je nou voor dat Oswaldo homo is. [9.1-10.8] MAIN_SPEAKER: Zou dat invloed hebben op je vriendschap? [10.9-12.5] MAIN_SPEAKER: Jazeker. Dat zou wel invloed hebben. [12.7-13.7] OTHER_SPEAKER_4: Wat bedoel je? [13.8-14.7] OTHER_SPEAKER_3: Gewoon, toch? [14.9-17.2] OTHER_SPEAKER_4: Dat heeft invloed op onze vriendschap. [17.3-20.5] OTHER_SPEAKER_3: Als ik zeg dat ik homo ben, dan ga je niet meer met me om. [20.7-22.8] OTHER_SPEAKER_3: Dan komt het wel op neer, ja. [22.9-25.1] OTHER_SPEAKER_3: Wat is er met me huis geweest? [25.3-27.8] OTHER_SPEAKER_2: Als jij nu zegt dat je homo bent, dan gaan mensen denken dat ik ook... [28.1-31.2] OTHER_SPEAKER_2: Omdat ik met jou nu, Safi, zeg nu, ik ben met jou naar huis geweest. [31.3-32.3] OTHER_SPEAKER_1: Kere! [32.5-33.5] OTHER_SPEAKER_1: Kere! [33.6-35.9] OTHER_SPEAKER_1: Dat krijgen we dan nooit doen! [36.1-38.1] OTHER_SPEAKER_1: Maar dat ligt gewoon aan jou, niet aan die mensen. [38.2-40.7] OTHER_SPEAKER_1: Nee, dan ligt het op dat moment aan jou, want jij bent dan homo. [40.9-42.8] MAIN_SPEAKER: Dat ligt gewoon aan jou, want jij bent dan homo. [42.9-45.0] MAIN_SPEAKER: Wacht even, wat is er aan de hand?"


,start,end,speaker,text,speaker_confidence,overlap,uncertain_assignment
0,0.8,3.3,MAIN_SPEAKER,"Oswaldo is jouw beste vriend, hè? Of niet?",1.00,False,False
1,3.4,4.4,MAIN_SPEAKER,"Hier in de klas, ja.",1.00,False,False
2,4.5,9.0,MAIN_SPEAKER,Stel je nou voor dat Oswaldo homo is.,1.00,False,False
3,9.1,10.8,MAIN_SPEAKER,Zou dat invloed hebben op je vriendschap?,1.00,False,False
4,10.9,12.5,MAIN_SPEAKER,Jazeker. Dat zou wel invloed hebben.,0.94,False,False
5,12.7,13.7,OTHER_SPEAKER_4,Wat bedoel je?,0.99,False,False
6,13.8,14.7,OTHER_SPEAKER_3,"Gewoon, toch?",1.00,False,False
7,14.9,17.2,OTHER_SPEAKER_4,Dat heeft invloed op onze vriendschap.,1.00,False,False
8,17.3,20.5,OTHER_SPEAKER_3,"Als ik zeg dat ik homo ben, dan ga je niet mee...",1.00,False,False
9,20.7,22.8,OTHER_SPEAKER_3,"Dan komt het wel op neer, ja.",0.95,False,True


,start,end,speaker,text,avg_logprob,no_speech_prob,compression_ratio,quality_flags,speaker_raw,speaker_confidence,overlap,uncertain_assignment
0,0.82,3.26,MAIN_SPEAKER,"Oswaldo is jouw beste vriend, hè? Of niet?",-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
1,3.38,4.38,MAIN_SPEAKER,"Hier in de klas, ja.",-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
2,4.50,8.98,MAIN_SPEAKER,Stel je nou voor dat Oswaldo homo is.,-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
3,9.10,10.82,MAIN_SPEAKER,Zou dat invloed hebben op je vriendschap?,-0.33186,0.043011,1.698842,[],SPEAKER_02,1.0000,False,False
4,10.94,12.54,MAIN_SPEAKER,Jazeker. Dat zou wel invloed hebben.,-0.33186,0.043011,1.698842,[],SPEAKER_02,0.9385,False,False
5,12.66,13.66,OTHER_SPEAKER_4,Wat bedoel je?,-0.33186,0.043011,1.698842,[],SPEAKER_04,0.9859,False,False
6,13.78,14.74,OTHER_SPEAKER_3,"Gewoon, toch?",-0.33186,0.043011,1.698842,[],SPEAKER_01,1.0000,False,False
7,14.86,17.22,OTHER_SPEAKER_4,Dat heeft invloed op onze vriendschap.,-0.33186,0.043011,1.698842,[],SPEAKER_04,1.0000,False,False
8,17.34,20.54,OTHER_SPEAKER_3,"Als ik zeg dat ik homo ben, dan ga je niet mee...",-0.33186,0.043011,1.698842,[],SPEAKER_01,1.0000,False,False
9,20.66,22.82,OTHER_SPEAKER_3,"Dan komt het wel op neer, ja.",-0.33186,0.043011,1.698842,[],SPEAKER_01,0.9516,False,True


### 4.2.1. Sprekerdiarisatie conclusie

In dit fragment onderscheidt de diarization meerdere sprekers en werkt deze daardoor beter dan in fragment 2. De globale sprekerstructuur van het gesprek wordt zichtbaar, maar door overlap, rumoer en snelle wisselingen blijven sommige labels onzeker.  
De diarization lijkt daarmee bruikbaar als ondersteunende context, maar niet als foutloze bron voor exacte sprekeridentificatie. Vooral bij overlappende spraak en korte uitingen moet rekening worden gehouden met verkeerde of onzekere toewijzingen.

## 5. Eindconclusie

De tests laten zien dat Whisper Medium voorlopig voldoende transcriptkwaliteit levert voor verdere ontwikkeling van de workshopcoach. In alle drie de fragmenten blijft de belangrijkste inhoud van gesprekken grotendeels behouden, ook wanneer sprake is van rumoer of meerdere sprekers. De kwaliteit neemt af bij overlap en chaotische gesprekken, maar docentvragen, uitleg en langere interventies blijven meestal herkenbaar. 
De grootste onzekerheid zit op dit moment niet in de algemene transcriptiekwaliteit, maar in de sprekerverdeling. Korte docent- en leerlingreacties kunnen binnen één Whisper-segment terechtkomen, waardoor één sprekerlabel aan tekst van meerdere personen wordt gekoppeld. Daarnaast laat fragment 2 zien dat diarization in sommige situaties vrijwel alle spraak aan de hoofdspreker kan toewijzen.

Voor de beoogde analyse betekent dit dat de pipeline voorlopig geschikt lijkt voor indicatoren die vooral gebaseerd zijn op duidelijke en langere docentuitspraken, zoals vraaggedrag, uitleg en expliciete controles op begrip. Indicatoren die sterk afhankelijk zijn van exacte beurtwisseling of korte leerlingreacties zijn gevoeliger voor fouten.
Op basis van deze tests is er op dit moment geen aanleiding om de transcriptiepipeline verder te optimaliseren voordat de eerste classificatie-experimenten zijn uitgevoerd. De gevonden beperkingen worden vastgelegd en kunnen opnieuw worden onderzocht wanneer blijkt dat ze de didactische classificatie daadwerkelijk belemmeren.